# 01 — GEE CCDC Temporal Segmentation, Feature Stack & Sample CSV Export

This Google Colab notebook walks through **Stages 1–2** of `LC_Framework` in Earth Engine:
1. Install `lc-framework[gee]` and authenticate Earth Engine
2. Load your `ProjectConfig` (or define an inline study area & `ClassSchema`)
3. Build harmonized Sentinel-2 / Landsat monthly medians and fit (or load) a CCDC `Temporal Segment` mosaic
4. Reconstruct wet/dry synthetic reflectance & harmonics and compose the multi-sensor feature stack (optical indices, phenological deltas, Sentinel-1 ARD, PALSAR-2, NASADEM terrain, GLCM texture)
5. Sample training points and export a clean **`.csv`** (with `class_id` + `class_name`) to Google Drive for local RFECV feature selection (`03_feature_selection_rfecv.ipynb`)

In [ ]:
# 1. Colab Bootstrap — install GEE extra only
# !pip install -q "lc-framework[gee] @ git+https://github.com/<your-org>/LC_Framework.git"

from pathlib import Path
import ee

from lc_framework.config import ClassDefinition, ClassSchema, ProjectConfig
from lc_framework.temporal import (
    CCDCModel,
    CCDCParams,
    CCDCProfile,
    export_ccdc_tiles,
    get_s2_collection,
    to_monthly_median,
)
from lc_framework.features import (
    FeatureStackSpec,
    build_feature_stack,
    export_sample_features_to_drive,
    extract_sample_features,
)

## 2. Authenticate GEE & Load Project Configuration

In [ ]:
# Replace with your Google Cloud / Earth Engine project ID
GEE_PROJECT = "your-gee-project-id"
# ee.Authenticate()
# ee.Initialize(project=GEE_PROJECT)

CONFIG_PATH = Path("../configs/example_project.yaml")
if CONFIG_PATH.exists():
    cfg = ProjectConfig.from_yaml(CONFIG_PATH)
    schema = cfg.class_schema
    seasons = cfg.seasons
else:
    cfg = None
    seasons = {"wet": 0.16, "dry": 0.65, "harmonic_mid": 0.50}
    schema = ClassSchema(
        classes=[
            ClassDefinition(id=1, name="Forest", color="#1b5e20"),
            ClassDefinition(id=2, name="Agroforestry", color="#66bb6a"),
            ClassDefinition(id=3, name="Cropland", color="#fbc02d"),
            ClassDefinition(id=4, name="Built-up", color="#e53935"),
            ClassDefinition(id=5, name="Waterbody", color="#0288d1"),
        ]
    )

TARGET_YEAR = 2025
print(f"Configured classes: {schema.class_names}")

## 3. Build Harmonized Sentinel-2 Time Series & CCDC `Temporal Segment` Model
For small AOIs you can fit CCDC on the fly; for national/regional extents, export CCDC tiles first with `export_ccdc_tiles(..., dry_run=False)` and reload them via `CCDCModel.from_tiles(...)`.

In [ ]:
# Define AOI (load from cfg.aoi_asset or specify a bounding box)
# aoi = ee.FeatureCollection(cfg.aoi_asset).geometry()
# s2_col = get_s2_collection(aoi, "2020-01-01", "2025-12-31", reflectance_scale=10000.0)
# monthly = to_monthly_median(s2_col, "2020-01-01", "2025-12-31")
#
# Option A: Fit on the fly (small AOIs)
# model = CCDCModel.fit(monthly, params=CCDCParams.sentinel2(), profile=CCDCProfile.standard())
#
# Option B: Preview / launch tiled CCDC asset export (large AOIs)
# plans, tasks = export_ccdc_tiles(
#     monthly,
#     bounds=(125.0, -9.0, 126.0, -8.0),
#     export_prefix=f"projects/{GEE_PROJECT}/assets/s2_ccdc",
#     tile_size_deg=0.5,
#     scale_m=10,
#     crs=cfg.crs if cfg else "EPSG:4326",
#     params=CCDCParams.sentinel2(),
#     dry_run=True,
# )
#
# Option C: Load pre-exported CCDC tiles
# model = CCDCModel.from_tiles(["projects/.../tile_r0_c0"], aoi=aoi)

## 4. Compose Multi-Sensor Feature Stack & Export Sample CSV for RFECV

In [ ]:
# wet_synth = model.synthetic(TARGET_YEAR + seasons["wet"])
# dry_synth = model.synthetic(TARGET_YEAR + seasons["dry"])
# harmonics = model.harmonics(TARGET_YEAR + seasons["harmonic_mid"])
#
# spec = FeatureStackSpec(
#     indices=("NDVI", "EVI", "MNDWI", "NDBI", "NBR", "BSI"),
#     include_deltas=True,
#     include_s1=True,
#     include_scansar=False,
#     include_palsar=True,
#     include_topography=True,
#     include_glcm=True,
# )
# feature_stack = build_feature_stack(
#     aoi=aoi,
#     year=TARGET_YEAR,
#     wet_image=wet_synth,
#     dry_image=dry_synth,
#     harmonics_image=harmonics,
#     spec=spec,
# )
#
# # Sample labeled points and export CSV to Drive -> ready for Notebook 03 (RFECV)
# samples_fc = ee.FeatureCollection(f"projects/{GEE_PROJECT}/assets/training_samples")
# sampled_fc = extract_sample_features(
#     feature_stack,
#     samples_fc,
#     class_schema=schema,
#     class_property="class_id",
#     scale_m=10.0,
# )
# export_sample_features_to_drive(
#     sampled_fc,
#     description=f"samples_features_{TARGET_YEAR}",
#     folder="lc_framework_exports",
#     dry_run=False,
# )